# Packages import

In [14]:
import os
import yaml
import requests
import pandas as pd
import re
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo Scraper

In [ ]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"

In [3]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
auth = HTTPBasicAuth(username, password)

In [4]:
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

200


In [5]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [6]:
group = page_dom.select_one("div.grupa").get_text()
print(group)

ZICSS1-1212


In [7]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag.prettify()))
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [9]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia","wykład","egzamin"])]

In [10]:
classes[['Day', 'Start Time', 'hyphen', 'End Time', 'Duration']] = classes['Dzień, godzina'].str.split(' ',expand=True)

In [11]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [12]:
classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [15]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.)*",
    r"\1",
    regex=True
)

In [16]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [17]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8")

In [ ]:
classes